In [7]:
# ==============================================================================
# SCRIPT DE TREINAMENTO - AUDITORIA AUTOMÁTICA DE COMPRESSORES (IA)
# Objetivo: Extrair dados históricos de planilhas Excel, tratar as variáveis
# termodinâmicas e treinar um modelo Random Forest para tomada de decisão.
# ==============================================================================

# Importação das bibliotecas essenciais
import pandas as pd                               # Para manipulação e análise de tabelas de dados
import glob                                       # Para buscar múltiplos arquivos em diretórios
import os                                         # Para manipulação de caminhos do sistema operacional
from sklearn.model_selection import train_test_split # (Opcional) Para dividir dados de treino e teste
from sklearn.ensemble import RandomForestClassifier  # Algoritmo de Inteligência Artificial escolhido
import joblib                                     # Para empacotar e salvar o modelo treinado em disco

# ==============================================================================
# 1. FUNÇÃO DE RADAR (MAPEAMENTO DINÂMICO DE COLUNAS)
# Função responsável por ler a estrutura do Excel e descobrir em quais colunas
# estão as variáveis, garantindo que o código funcione mesmo se o template mudar.
# ==============================================================================
def mapear_colunas(df_raw):
    mapa = {} # Dicionário que guardará o nome da variável e o índice da coluna
    
    # Passo A: Busca baseada em Cabeçalhos
    # Varre todas as colunas da planilha para encontrar os nomes das métricas
    for col in range(df_raw.shape[1]):
        # Extrai as 15 primeiras linhas da coluna, converte para minúsculo e junta em um único texto
        # Isso cria uma "assinatura" da coluna para buscarmos palavras-chave, ignorando células vazias (NaN)
        texto_coluna = " ".join([str(x).lower() for x in df_raw.iloc[0:15, col].tolist()])
        
        # Mapeamento da coluna de Decisão Final
        if 'decision' in texto_coluna and 'final' in texto_coluna:
            mapa['Decisao'] = col
            
        # Mapeamento das colunas de Capacidade Frigorífica (Qo) das amostras A e B
        if 'qo [w]' in texto_coluna:
            mapa['Qo_A'] = col; mapa['Qo_B'] = col + 1
            
        # Mapeamento das colunas de Potência Consumida (Wc) das amostras A e B
        if 'wc [w]' in texto_coluna:
            mapa['Wc_A'] = col; mapa['Wc_B'] = col + 1
            
        # Mapeamento das colunas de Eficiência (COP) das amostras A e B
        if 'cop [-]' in texto_coluna:
            mapa['COP_A'] = col; mapa['COP_B'] = col + 1
            
        # Mapeamento da Rotação (RPM), garantindo que não confunda com a coluna de desvio (deviation)
        if 'speed [rpm]' in texto_coluna and 'deviation' not in texto_coluna:
            mapa['RPM_A'] = col; mapa['RPM_B'] = col + 1
            
        # Mapeamento do bloco de desvios percentuais (Variação entre as amostras)
        if 'deviation to avg' in texto_coluna:
            mapa['Var_Qo'] = col; mapa['Var_Wc'] = col + 1; mapa['Var_COP'] = col + 2; mapa['Var_RPM'] = col + 3

    # Passo B: Busca baseada em Dados (Bypass de cabeçalhos mesclados)
    # Procura a coluna de "OK" farejando os dados reais (linhas 13 a 30) nas colunas centrais (30 a 60)
    # Passo B: Farejador de OK e Condição (Bypass de cabeçalhos mesclados)
    for col in range(30, min(60, df_raw.shape[1])):
        # Ampliamos a amostra para olhar da linha 10 até a 30 (índices 9 a 29 do Pandas)
        amostra_dados = df_raw.iloc[9:30, col].astype(str).str.strip().str.upper().tolist()
        
        if 'OK' in amostra_dados:
            mapa['Flag_OK'] = col
            mapa['Condicao'] = col + 1  # A Condição é sempre a vizinha da direita!
            
            # NOVIDADE: Descobre a linha exata onde os dados começam!
            # amostra_dados.index('OK') acha a posição do 1º 'OK' dentro da amostra.
            # Como a amostra começou no índice 9, somamos 9 para ter a linha real.
            mapa['Indice_Inicio_Dados'] = 9 + amostra_dados.index('OK')
            break
            
    return mapa

# ==============================================================================
# 2. EXTRAÇÃO E CONSOLIDAÇÃO DO HISTÓRICO DE PLANILHAS
# Varre o diretório, aplica o radar em cada arquivo e compila uma base única.
# ==============================================================================

# Localiza todos os arquivos Excel (suportando tanto planilhas com macro quanto normais)
arquivos_xlsm = glob.glob('historico_testes/*.xlsm')
arquivos_xlsx = glob.glob('historico_testes/*.xlsx')
arquivos_excel = arquivos_xlsm + arquivos_xlsx

# Blindagem: Ignora arquivos temporários (invisíveis) gerados pelo Windows quando uma planilha está aberta
arquivos_excel = [arquivo for arquivo in arquivos_excel if not os.path.basename(arquivo).startswith('~$')]

lista_de_tabelas = [] # Lista que armazenará os DataFrames individuais antes da concatenação

for arquivo in arquivos_excel:
    # Lê a aba 'CAL' do Excel de forma bruta (sem cabeçalhos pré-definidos)
    df_raw = pd.read_excel(arquivo, sheet_name='CAL', header=None)
    
    # Aciona a função Radar para descobrir as coordenadas das colunas neste arquivo específico
    mapa = mapear_colunas(df_raw)
    
    # Validação de integridade: Pula a planilha se o radar não encontrar as colunas mínimas necessárias
    if 'Condicao' not in mapa or 'Decisao' not in mapa:
        print(f"⚠️ Radar falhou em {os.path.basename(arquivo)}. Faltam colunas vitais. Pulando...")
        continue
        
    # Constrói um novo DataFrame apenas com as colunas mapeadas pelo radar
    df_limpo = pd.DataFrame()
    df_limpo['Flag_AW'] = df_raw.iloc[:, mapa['Flag_OK']]
    df_limpo['Condicao_Ensaio'] = df_raw.iloc[:, mapa['Condicao']]
    df_limpo['Qo_A'] = df_raw.iloc[:, mapa['Qo_A']]
    df_limpo['Qo_B'] = df_raw.iloc[:, mapa['Qo_B']]
    df_limpo['Wc_A'] = df_raw.iloc[:, mapa['Wc_A']]
    df_limpo['Wc_B'] = df_raw.iloc[:, mapa['Wc_B']]
    df_limpo['COP_A'] = df_raw.iloc[:, mapa['COP_A']]
    df_limpo['COP_B'] = df_raw.iloc[:, mapa['COP_B']]
    df_limpo['RPM_A'] = df_raw.iloc[:, mapa['RPM_A']]
    df_limpo['RPM_B'] = df_raw.iloc[:, mapa['RPM_B']]
    df_limpo['Var_Qo'] = df_raw.iloc[:, mapa['Var_Qo']]
    df_limpo['Var_Wc'] = df_raw.iloc[:, mapa['Var_Wc']]
    df_limpo['Var_COP'] = df_raw.iloc[:, mapa['Var_COP']]
    df_limpo['Var_RPM'] = df_raw.iloc[:, mapa['Var_RPM']]
    df_limpo['Decisao_Final'] = df_raw.iloc[:, mapa['Decisao']]
    df_limpo['Origem_Arquivo'] = os.path.basename(arquivo)
    
    # Recorta as 12 primeiras linhas (cabeçalhos originais da planilha) para manter apenas os dados numéricos
    df_limpo = df_limpo.iloc[12:].reset_index(drop=True)
    
    # Filtro de Validade: Mantém estritamente as linhas onde o engenheiro validou com "OK"
    df_limpo = df_limpo[df_limpo['Flag_AW'].astype(str).str.strip().str.upper() == 'OK']
    
    # Remove a coluna de Flag, pois ela não é uma variável termodinâmica e não deve entrar no treino
    df_limpo = df_limpo.drop(columns=['Flag_AW'])
    
    # Trava de segurança: Adiciona à lista final apenas se sobrou algum dado após os filtros
    if not df_limpo.empty:
        lista_de_tabelas.append(df_limpo)

# Validação final de extração
if len(lista_de_tabelas) == 0:
    print("ERRO CRÍTICO: Nenhuma planilha tinha dados com 'OK'. Verifique os arquivos.")
else:
    # Junta todas as planilhas validadas em um único DataFrame mestre
    df_analise = pd.concat(lista_de_tabelas, ignore_index=True)
    print(f"Buscador extraiu dados com sucesso de {len(lista_de_tabelas)} planilhas!")

# ==============================================================================
# 3. ENGENHARIA DE RECURSOS (FEATURE ENGINEERING)
# Preparação dos dados para a IA: Transformação de strings em variáveis categóricas
# ==============================================================================

# A coluna Condicao_Ensaio contém múltiplos valores em uma única string (Ex: "-25 25 16 4000")
# O método split() separa esses valores usando o espaço em branco como delimitador
cond_sep = df_analise['Condicao_Ensaio'].astype(str).str.split(expand=True)

# Atribui os valores separados às suas respectivas variáveis termodinâmicas
df_analise['Tev'] = cond_sep[0]   # Temperatura de Evaporação
df_analise['Tcd'] = cond_sep[1]   # Temperatura de Condensação

# Valida se as colunas opcionais existem na string antes de atribuí-las
df_analise['Tsuc'] = cond_sep[2] if 2 in cond_sep.columns else None  # Temperatura de Sucção
df_analise['speed'] = cond_sep[3] if 3 in cond_sep.columns else None # Velocidade (RPM)

# Identifica quais dessas colunas de ambiente foram efetivamente criadas
colunas_existentes = [col for col in ['Tev', 'Tcd', 'Tsuc', 'speed'] if col in df_analise.columns]

# Converte as variáveis categóricas (texto/fatores) em colunas binárias (One-Hot Encoding)
# Isso permite que a IA entenda as temperaturas e rotações de forma matemática (0 ou 1)
df_analise = pd.get_dummies(df_analise, columns=colunas_existentes, dummy_na=False)

# Remove a coluna original de texto aglomerado, pois a IA não consegue processar strings puras
df_analise = df_analise.drop(columns=['Condicao_Ensaio'])

# ==============================================================================
# 4. TREINAMENTO DA INTELIGÊNCIA ARTIFICIAL (MODO PRODUÇÃO)
# Treinamento do modelo definitivo utilizando 100% dos dados estruturados.
# ==============================================================================

# Separação das variáveis: X (Features/Sintomas) e y (Target/Diagnóstico)
grupos = df_analise['Origem_Arquivo']
X = df_analise.drop(columns=['Decisao_Final', 'Origem_Arquivo'])
y = df_analise['Decisao_Final']

# Instancia o algoritmo Random Forest (Floresta Aleatória) com 100 árvores de decisão
# Parâmetro 'class_weight="balanced"': Crucial para lidar com classes raras. 
# Ele pune severamente a IA caso ela erre decisões minoritárias (A ou B) por vício estatístico no 'AVG'
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    random_state=42,          # Garante a reprodutibilidade (o resultado será o mesmo toda vez que rodar)
    class_weight='balanced'   # Antídoto contra o desbalanceamento das classes
)

# Aciona o processo de aprendizado: A IA mapeia os padrões em 100% da base
modelo_rf.fit(X, y)

# ==============================================================================
# 5. EXPORTAÇÃO DO CÉREBRO DA IA
# Salva o modelo treinado para ser consumido pelo aplicativo Desktop (app.exe).
# ==============================================================================

# Salva o motor de inferência (as regras matemáticas aprendidas)
joblib.dump(modelo_rf, 'cerebro_ia.pkl')

# Salva a estrutura exata de colunas que a IA usou para treinar.
# Isso garante que no futuro, quando o app ler novos dados, a ordem e o formato das colunas não quebrem o modelo
joblib.dump(list(X.columns), 'colunas_ia.pkl')

# Relatório final de execução
print("\n==============================================================================")
print(" RELATÓRIO FINAL DE TREINAMENTO")
print("==============================================================================")
print(f"📊 Total de Planilhas processadas: {len(lista_de_tabelas)}")
print(f"📈 Total de Linhas (exemplos) absorvidos: {len(df_analise)}")
print("✅ Processo finalizado: O motor da IA foi treinado e atualizado em disco!")
print("==============================================================================\n")

Buscador extraiu dados com sucesso de 10 planilhas!

 RELATÓRIO FINAL DE TREINAMENTO
📊 Total de Planilhas processadas: 10
📈 Total de Linhas (exemplos) absorvidos: 515
✅ Processo finalizado: O motor da IA foi treinado e atualizado em disco!



In [8]:
import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("=================================================================")
print(" 1. AMBIENTE, CLASSES E UNIDADES")
print("=================================================================")
print(f"Versão do scikit-learn: {sklearn.__version__}")
print(f"Total de registros: {len(X)} | Total de planilhas: {grupos.nunique()}")
print("\nDistribuição das classes na base (515 registros):")
print(y.value_counts())

print("\nMapeamento de classes (índice da árvore -> rótulo real):")
for idx, classe in enumerate(modelo_rf.classes_):
    print(f"  class {float(idx)} -> {classe}")

print("\nVerificação de escala (Var_Qo e Var_COP):")
print(X[['Var_Qo', 'Var_COP', 'Var_Wc']].apply(pd.to_numeric, errors='coerce').describe().loc[['min', 'mean', 'max']])

print("\n=================================================================")
print(" 2. VALIDAÇÃO CRUZADA (StratifiedGroupKFold, k=5)")
print("=================================================================")
sgkf = StratifiedGroupKFold(n_splits=5)

acuracias, precisoess, recalls, f1s = [], [], [], []
y_true_total, y_pred_total = [], []

for fold, (idx_treino, idx_teste) in enumerate(sgkf.split(X, y, groups=grupos), 1):
    X_tr, X_te = X.iloc[idx_treino], X.iloc[idx_teste]
    y_tr, y_te = y.iloc[idx_treino], y.iloc[idx_teste]
    
    # Confirmação de vazamento zero
    arqs_tr = set(grupos.iloc[idx_treino])
    arqs_te = set(grupos.iloc[idx_teste])
    assert arqs_tr.isdisjoint(arqs_te), "ERRO: Vazamento de arquivo detectado!"
    
    clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_te, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    
    acuracias.append(acc); precisoess.append(prec); recalls.append(rec); f1s.append(f1)
    y_true_total.extend(y_te); y_pred_total.extend(y_pred)
    
    print(f"Fold {fold} | Treino: {len(idx_treino)} | Teste: {len(idx_teste)} ({len(arqs_te)} planilhas) "
          f"| Acc: {acc*100:.2f}% | Prec(macro): {prec*100:.2f}% | Rec(macro): {rec*100:.2f}% | F1(macro): {f1*100:.2f}%")

print("\n--- RESUMO K-FOLD (MÉDIA ± DESVIO-PADRÃO) ---")
print(f"Acurácia:         {np.mean(acuracias)*100:.2f}% ± {np.std(acuracias)*100:.2f}%")
print(f"Precisão (macro): {np.mean(precisoess)*100:.2f}% ± {np.std(precisoess)*100:.2f}%")
print(f"Recall (macro):   {np.mean(recalls)*100:.2f}% ± {np.std(recalls)*100:.2f}%")
print(f"F1-Score (macro): {np.mean(f1s)*100:.2f}% ± {np.std(f1s)*100:.2f}%")

print("\n--- MATRIZ DE CONFUSÃO ABSOLUTA (CONSOLIDADA DOS 5 FOLDS) ---")
labels = list(modelo_rf.classes_)
cm = confusion_matrix(y_true_total, y_pred_total, labels=labels)
df_cm = pd.DataFrame(cm, index=[f"Real: {c}" for c in labels], columns=[f"Prev: {c}" for c in labels])
print(df_cm)

 1. AMBIENTE, CLASSES E UNIDADES
Versão do scikit-learn: 1.9.1
Total de registros: 515 | Total de planilhas: 10

Distribuição das classes na base (515 registros):
Decisao_Final
AVG        499
B            9
discard      4
A            3
Name: count, dtype: int64

Mapeamento de classes (índice da árvore -> rótulo real):
  class 0.0 -> A
  class 1.0 -> AVG
  class 2.0 -> B
  class 3.0 -> discard

Verificação de escala (Var_Qo e Var_COP):
            Var_Qo   Var_COP    Var_Wc
min   3.648803e-07  0.000029  0.000067
mean  1.137866e-02  0.011691  0.008482
max   2.924124e-01  0.281348  0.300026

 2. VALIDAÇÃO CRUZADA (StratifiedGroupKFold, k=5)


c:\Users\lucasa05\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Fold 1 | Treino: 411 | Teste: 104 (2 planilhas) | Acc: 97.12% | Prec(macro): 24.75% | Rec(macro): 24.75% | F1(macro): 24.75%
Fold 2 | Treino: 406 | Teste: 109 (2 planilhas) | Acc: 94.50% | Prec(macro): 48.57% | Rec(macro): 45.00% | F1(macro): 46.49%
Fold 3 | Treino: 413 | Teste: 102 (2 planilhas) | Acc: 96.08% | Prec(macro): 66.67% | Rec(macro): 65.31% | F1(macro): 65.97%
Fold 4 | Treino: 415 | Teste: 100 (2 planilhas) | Acc: 99.00% | Prec(macro): 50.00% | Rec(macro): 49.50% | F1(macro): 49.75%
Fold 5 | Treino: 415 | Teste: 100 (2 planilhas) | Acc: 100.00% | Prec(macro): 100.00% | Rec(macro): 100.00% | F1(macro): 100.00%

--- RESUMO K-FOLD (MÉDIA ± DESVIO-PADRÃO) ---
Acurácia:         97.34% ± 1.98%
Precisão (macro): 58.00% ± 24.89%
Recall (macro):   56.91% ± 25.13%
F1-Score (macro): 57.39% ± 25.03%

--- MATRIZ DE CONFUSÃO ABSOLUTA (CONSOLIDADA DOS 5 FOLDS) ---
               Prev: A  Prev: AVG  Prev: B  Prev: discard
Real: A              0          2        1              0
Real: AVG 

In [9]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(modelo_rf)
shap_values = explainer.shap_values(X.astype(float))

# Trata o formato de retorno do SHAP para classificação multiclasse
# (compatível com versões novas e antigas da biblioteca shap)
if isinstance(shap_values, list):
    # Lista de arrays [n_amostras, n_features] por classe
    importancia_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    # Array 3D (n_amostras, n_features, n_classes)
    importancia_shap = np.abs(shap_values).mean(axis=(0, 2))
else:
    importancia_shap = np.abs(shap_values).mean(axis=0)

peso_pct = (importancia_shap / importancia_shap.sum()) * 100
df_shap = pd.DataFrame({'Variavel': X.columns, 'SHAP_Medio_Abs': importancia_shap, 'Participacao (%)': peso_pct})
df_shap = df_shap.sort_values(by='Participacao (%)', ascending=False).reset_index(drop=True)

print("=========================================================")
print(" RANKING REAL DE IMPORTÂNCIA VIA SHAP (TreeExplainer)")
print("=========================================================")
print(df_shap.head(10).to_string(index=False, float_format="%.4f"))

 RANKING REAL DE IMPORTÂNCIA VIA SHAP (TreeExplainer)
Variavel  SHAP_Medio_Abs  Participacao (%)
 Var_COP          0.0761           17.9894
  Var_Qo          0.0610           14.4176
   COP_B          0.0360            8.5015
 Var_RPM          0.0257            6.0739
   COP_A          0.0240            5.6781
    Qo_A          0.0211            4.9817
    Qo_B          0.0186            4.3893
  Var_Wc          0.0183            4.3291
   RPM_B          0.0165            3.8911
   RPM_A          0.0141            3.3228


In [11]:
import pandas as pd
from sklearn.tree import export_text

print("=========================================================")
print(" 1. RANKING DE IMPORTÂNCIA DAS VARIÁVEIS (O PESO DA DECISÃO)")
print("=========================================================")
# Puxa o peso matemático de cada coluna na decisão final
importancias = modelo_rf.feature_importances_
colunas = X.columns

# Monta uma tabelinha bonita para leitura
df_importancia = pd.DataFrame({'Variavel': colunas, 'Peso (%)': importancias * 100})
df_importancia = df_importancia.sort_values(by='Peso (%)', ascending=False).reset_index(drop=True)

# Imprime o top 10
print(df_importancia.head(10).to_string(index=False, float_format="%.2f%%"))
print("\n")


print("=========================================================")
print(" 2. EXEMPLO DE REGRAS DE UMA ÁRVORE (MATRIZ DE DECISÃO)")
print("=========================================================")
# Extrai as regras em texto da Árvore Número 0 (a primeira das 100)
arvore_exemplo = modelo_rf.estimators_[0]
regras_texto = export_text(arvore_exemplo, feature_names=list(colunas), max_depth=3)

print("Nota: Este é apenas um recorte resumido de 1 das 100 árvores da IA.\n")
print(regras_texto)

 1. RANKING DE IMPORTÂNCIA DAS VARIÁVEIS (O PESO DA DECISÃO)
Variavel  Peso (%)
 Var_COP    14.92%
   COP_B     9.14%
  Var_Qo     8.55%
    Qo_B     6.55%
 Var_RPM     6.43%
    Qo_A     6.39%
  Var_Wc     5.77%
   COP_A     5.54%
    Wc_A     5.36%
    Wc_B     4.79%


 2. EXEMPLO DE REGRAS DE UMA ÁRVORE (MATRIZ DE DECISÃO)
Nota: Este é apenas um recorte resumido de 1 das 100 árvores da IA.

|--- Var_Wc <= 0.00
|   |--- RPM_B <= 5368.91
|   |   |--- class: 1.0
|   |--- RPM_B >  5368.91
|   |   |--- COP_B <= 1.88
|   |   |   |--- class: 1.0
|   |   |--- COP_B >  1.88
|   |   |   |--- class: 0.0
|--- Var_Wc >  0.00
|   |--- Var_COP <= 0.04
|   |   |--- Var_Wc <= 0.01
|   |   |   |--- Tev_2025-09-10 <= 0.50
|   |   |   |   |--- truncated branch of depth 8
|   |   |   |--- Tev_2025-09-10 >  0.50
|   |   |   |   |--- class: 1.0
|   |   |--- Var_Wc >  0.01
|   |   |   |--- class: 1.0
|   |--- Var_COP >  0.04
|   |   |--- Wc_B <= 103.29
|   |   |   |--- class: 0.0
|   |   |--- Wc_B >  103.2